## 1. Importar pandas y definir la ruta del archivo

In [7]:
import pandas as pd

df = pd.read_csv('Notas_de_estudiantes.csv')
df.head()

,nombre,apellido,semestre,edad,nota
0,Natalia,Mendoza,1ro,25,"6,09"
1,pedro,A guilar,1,24,"""2,51"""
2,Ramon,Rivera,-,25,"02,93"
3,Gabriela,Flore,8,85,"5,71"
4,Carlos,Martinez,3,,NaN


In [4]:
#Revision de datos
print("Resumen de estructura:")
df.info()

print("\nTipos de datos:")
display(df.dtypes)

print("\nValores nulos por columna:")
display(df.isna().sum())



Resumen de estructura:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   nombre    191 non-null    object
 1   apellido  196 non-null    object
 2   semestre  184 non-null    object
 3   edad      190 non-null    object
 4   nota      192 non-null    object
dtypes: object(5)
memory usage: 7.9+ KB

Tipos de datos:


nombre      object
apellido    object
semestre    object
edad        object
nota        object
dtype: object


Valores nulos por columna:


nombre       9
apellido     4
semestre    16
edad        10
nota         8
dtype: int64

## 2. Identificación de problemas de calidad
a) Valores faltantes
b) Duplicados
c) Outliers en `nota` (< 1.0 o > 7.0)

In [6]:
# Copia para evaluación de calidad
df_eval = df.copy()

# Normalizar texto y marcar tokens comunes de faltantes
for col in df_eval.columns:
    df_eval[col] = df_eval[col].astype("string").str.strip()

tokens_faltantes = {"", "na", "n/a", "null", "none"}
for col in df_eval.columns:
    lower_col = df_eval[col].str.lower()
    df_eval[col] = df_eval[col].where(~lower_col.isin(tokens_faltantes), pd.NA)

# a) Valores faltantes
faltantes_por_columna = df_eval.isna().sum()
total_faltantes = int(faltantes_por_columna.sum())

print("a) Valores faltantes")
print("Total de valores faltantes:", total_faltantes)
display(faltantes_por_columna)

# b) Duplicados
duplicados_filas = int(df_eval.duplicated().sum())
duplicados_estudiante = df_eval.duplicated(subset=["nombre", "apellido"], keep=False)
cant_duplicados_estudiante = int(duplicados_estudiante.sum())

print("\nb) Duplicados")
print("Filas totalmente duplicadas:", duplicados_filas)
print("Registros con nombre+apellido repetido:", cant_duplicados_estudiante)
if cant_duplicados_estudiante > 0:
    display(df_eval.loc[duplicados_estudiante, ["nombre", "apellido", "semestre", "edad", "nota"]].sort_values(["nombre", "apellido"]))

# c) Outliers en nota (<1.0 o >7.0)
nota_limpia = (
    df_eval["nota"]
    .str.replace('"', "", regex=False)
    .str.replace("pts", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.extract(r"([0-9]*\.?[0-9]+)", expand=False)
)
df_eval["nota_num"] = pd.to_numeric(nota_limpia, errors="coerce")

outliers_nota = df_eval[(df_eval["nota_num"] < 1.0) | (df_eval["nota_num"] > 7.0)]

print("\nc) Outliers en nota (<1.0 o >7.0)")
print("Cantidad de outliers:", len(outliers_nota))
if len(outliers_nota) > 0:
    display(outliers_nota[["nombre", "apellido", "nota", "nota_num"]].sort_values("nota_num"))

a) Valores faltantes
Total de valores faltantes: 67


nombre      16
apellido     9
semestre    17
edad        15
nota        10
dtype: int64


b) Duplicados
Filas totalmente duplicadas: 25
Registros con nombre+apellido repetido: 64


,nombre,apellido,semestre,edad,nota
62,Andres,Vargas,7,21,"""9,17"""
77,Andres,Vargas,4,20,"7,34"
97,Beatriz,diaz,primero,26,999
187,Beatriz,diaz,primero,26,999
50,Camila,Perez,9,25,"5,99"
...,...,...,...,...,...
66,<NA>,morales,8,22,"8,44"
67,<NA>,morales,8,22,"8,44"
6,<NA>,<NA>,3,23,"""3,29 pts"""
63,<NA>,<NA>,2,-5,"$5,82"



c) Outliers en nota (<1.0 o >7.0)
Cantidad de outliers: 82


,nombre,apellido,nota,nota_num
181,Laura,Vargas,0,0.0
163,Claudia,Sanchez,"$0,1",0.1
7,Beatriz,Gomez,"0,41",0.41
132,Fernando,Paredes,"00,41",0.41
135,Fernando,Paredes,"00,41",0.41
...,...,...,...,...
97,Beatriz,diaz,999,999.0
24,<NA>,Gomez,999,999.0
187,Beatriz,diaz,999,999.0
23,Luis,Lopez,1000,1000.0


## 3. Aplicación de métodos de limpieza
Se aplican tres métodos:
1. Imputación de faltantes con mediana (columnas numéricas).
2. Eliminación de filas duplicadas.
3. Corrección de outliers en `nota` ajustando al rango [1.0, 7.0].

In [8]:
# Pipeline de limpieza aplicado
df_limpio = df.copy()

# 1) Normalización y faltantes explícitos
for col in df_limpio.columns:
    df_limpio[col] = df_limpio[col].astype("string").str.strip()

tokens_faltantes = {"", "na", "n/a", "null", "none"}
for col in df_limpio.columns:
    df_limpio[col] = df_limpio[col].mask(df_limpio[col].str.lower().isin(tokens_faltantes), pd.NA)

# Convertir columnas numéricas
nota_num = (
    df_limpio["nota"]
    .str.replace('"', "", regex=False)
    .str.replace("pts", "", regex=False)
    .str.replace("$", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.extract(r"([0-9]*\.?[0-9]+)", expand=False)
)
df_limpio["nota_num"] = pd.to_numeric(nota_num, errors="coerce")
df_limpio["edad_num"] = pd.to_numeric(df_limpio["edad"], errors="coerce")

# Métricas antes de limpiar
faltantes_antes = int(df_limpio.isna().sum().sum())
duplicados_antes = int(df_limpio.duplicated().sum())
outliers_antes = int(((df_limpio["nota_num"] < 1.0) | (df_limpio["nota_num"] > 7.0)).sum())

# 2) Imputación de faltantes con mediana
df_limpio["edad_num"] = df_limpio["edad_num"].fillna(df_limpio["edad_num"].median())
df_limpio["nota_num"] = df_limpio["nota_num"].fillna(df_limpio["nota_num"].median())

# 3) Eliminar duplicados
df_limpio = df_limpio.drop_duplicates().copy()

# 4) Corregir outliers ajustando al rango [1, 7]
df_limpio["nota_num"] = df_limpio["nota_num"].clip(lower=1.0, upper=7.0)

# Métricas después de limpiar
faltantes_despues = int(df_limpio[["edad_num", "nota_num"]].isna().sum().sum())
duplicados_despues = int(df_limpio.duplicated().sum())
outliers_despues = int(((df_limpio["nota_num"] < 1.0) | (df_limpio["nota_num"] > 7.0)).sum())

print("Resumen de limpieza aplicada")
print("- Faltantes (edad_num + nota_num) antes:", faltantes_antes)
print("- Faltantes (edad_num + nota_num) después:", faltantes_despues)
print("- Duplicados antes:", duplicados_antes)
print("- Duplicados después:", duplicados_despues)
print("- Outliers de nota antes:", outliers_antes)
print("- Outliers de nota después:", outliers_despues)

display(df_limpio[["nombre", "apellido", "edad_num", "nota_num"]].head())

Resumen de limpieza aplicada
- Faltantes (edad_num + nota_num) antes: 108
- Faltantes (edad_num + nota_num) después: 0
- Duplicados antes: 25
- Duplicados después: 0
- Outliers de nota antes: 82
- Outliers de nota después: 0


,nombre,apellido,edad_num,nota_num
0,Natalia,Mendoza,25,6.09
1,pedro,A guilar,24,2.51
2,Ramon,Rivera,25,2.93
3,Gabriela,Flore,85,5.71
4,Carlos,Martinez,24,5.6


## 4. Comentario sobre decisiones de limpieza
**a) Valores faltantes**: se imputaron con la **mediana** en `edad_num` y `nota_num`.
Se eligió mediana porque es más robusta que la media frente a valores extremos (por ejemplo, notas como 999 o 1000) y reduce el sesgo en los datos.

**b) Duplicados**: se eliminaron filas duplicadas exactas con `drop_duplicates()`.
Se tomó esta decisión para evitar contar el mismo registro más de una vez y no distorsionar estadísticas como promedio, distribución de notas o cantidad de estudiantes.

**c) Outliers en nota**: se corrigieron ajustando los valores al rango válido **[1.0, 7.0]** usando `clip`.
Se prefirió ajustar en lugar de eliminar para conservar la mayor cantidad de registros posible, manteniendo el criterio académico esperado para la escala de notas.